In [44]:
import sys
import warnings
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm
from scipy.optimize import minimize
sys.path.insert(0, "../../Utilities/")
import common_functions as cf

warnings.filterwarnings('ignore')

In [37]:
data_in = np.load('initial_inputs.npy')
data_out = np.load('initial_outputs.npy')
print(data_in)
print(data_out)

[[0.27262382 0.32449536 0.89710881 0.83295115 0.15406269 0.79586362]
 [0.54300258 0.9246939  0.34156746 0.64648585 0.71844033 0.34313266]
 [0.09083225 0.66152938 0.06593091 0.25857701 0.96345285 0.6402654 ]
 [0.11886697 0.61505494 0.90581639 0.8553003  0.41363143 0.58523563]
 [0.63021764 0.8380969  0.68001305 0.73189509 0.52673671 0.34842921]
 [0.76491917 0.25588292 0.60908422 0.21807904 0.32294277 0.09579366]
 [0.05789554 0.49167222 0.24742222 0.21811844 0.42042833 0.73096984]
 [0.19525188 0.07922665 0.55458046 0.17056682 0.01494418 0.10703171]
 [0.64230298 0.83687455 0.02179269 0.10148801 0.68307083 0.6924164 ]
 [0.78994255 0.19554501 0.57562333 0.07365919 0.25904917 0.05109986]
 [0.52849733 0.45742436 0.36009569 0.36204551 0.81689098 0.63747637]
 [0.72261522 0.01181284 0.06364591 0.16517311 0.07924415 0.35995166]
 [0.07566492 0.33450212 0.13273274 0.60831236 0.91838592 0.82233079]
 [0.94245084 0.37743962 0.48612233 0.22879108 0.08263175 0.71195755]
 [0.14864702 0.03394336 0.72880565

Week-01

In [38]:
X_init = np.array([
 [0.27262382, 0.32449536, 0.89710881, 0.83295115, 0.15406269, 0.79586362],
 [0.54300258, 0.9246939 , 0.34156746, 0.64648585, 0.71844033, 0.34313266],
 [0.09083225, 0.66152938, 0.06593091, 0.25857701, 0.96345285, 0.6402654 ],
 [0.11886697, 0.61505494, 0.90581639, 0.8553003 , 0.41363143, 0.58523563],
 [0.63021764, 0.8380969 , 0.68001305, 0.73189509, 0.52673671, 0.34842921],
 [0.76491917, 0.25588292, 0.60908422, 0.21807904, 0.32294277, 0.09579366],
 [0.05789554, 0.49167222, 0.24742222, 0.21811844, 0.42042833, 0.73096984],
 [0.19525188, 0.07922665, 0.55458046, 0.17056682, 0.01494418, 0.10703171],
 [0.64230298, 0.83687455, 0.02179269, 0.10148801, 0.68307083, 0.6924164 ],
 [0.78994255, 0.19554501, 0.57562333, 0.07365919, 0.25904917, 0.05109986],
 [0.52849733, 0.45742436, 0.36009569, 0.36204551, 0.81689098, 0.63747637],
 [0.72261522, 0.01181284, 0.06364591, 0.16517311, 0.07924415, 0.35995166],
 [0.07566492, 0.33450212, 0.13273274, 0.60831236, 0.91838592, 0.82233079],
 [0.94245084, 0.37743962, 0.48612233, 0.22879108, 0.08263175, 0.71195755],
 [0.14864702, 0.03394336, 0.72880565, 0.31606646, 0.02176938, 0.51691776],
 [0.81711239, 0.54816823, 0.10334758, 0.12436955, 0.72823482, 0.44967361],
 [0.41762629, 0.06409998, 0.24566877, 0.5590408 , 0.19153138, 0.25464092],
 [0.72628566, 0.46489581, 0.92457051, 0.8072454 , 0.6354384 , 0.14341787],
 [0.31981043, 0.52009759, 0.29067775, 0.87670668, 0.49503469, 0.6190825 ],
 [0.87987128, 0.39796199, 0.00363456, 0.95699064, 0.26451373, 0.11486924],
 [0.54124078, 0.63140314, 0.03190205, 0.44998156, 0.79865282, 0.63370429],
 [0.22634792, 0.11502581, 0.82474966, 0.94538372, 0.90531153, 0.95101392],
 [0.68685257, 0.04101721, 0.00757301, 0.285009  , 0.69156848, 0.6555429 ],
 [0.17597754, 0.6244165 , 0.29554198, 0.46955276, 0.09776977, 0.72814108],
 [0.88164674, 0.20445019, 0.41447436, 0.42038468, 0.26491501, 0.73066019],
 [0.06661051, 0.52804507, 0.8160952 , 0.96101714, 0.08650933, 0.77778822],
 [0.93246638, 0.48881189, 0.25860774, 0.95624344, 0.19042781, 0.51985176],
 [0.84686697, 0.14242917, 0.06066859, 0.75629213, 0.5523983 , 0.08130609],
 [0.80628208, 0.32412237, 0.72607601, 0.14871213, 0.7193764 , 0.36288398],
 [0.47682313, 0.34094195, 0.01433523, 0.88013956, 0.9986547 , 0.07966402],
])
 

y_init = np.array([
 0.6044327 , 0.56275307, 0.00750324, 0.0614243 , 0.2730468 , 0.08374657,
 1.3649683 , 0.09264495, 0.0178696 , 0.03356494, 0.0735163 , 0.2063097,
 0.00882563, 0.26840032, 0.61152553, 0.01479818, 0.27489251, 0.06676325,
 0.04211835, 0.00270147, 0.01820907, 0.00701603, 0.10050661, 0.47539552,
 0.67514163, 0.51645722, 0.00377748, 0.00313433, 0.02134252, 0.09541116
])

X_init = data_in
y_init = data_out

y_best = y_init.max()  # best performance so far

kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_init)

# --- Expected Improvement acquisition function ---
def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    mu, sigma = mu[0], sigma[0]
    if sigma == 0.0:
        return 0.0
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # negative for minimization in scipy

# --- Optimize acquisition function ---
bounds = [(0,1)]*6
best_x = None
best_ei = float('inf')

for _ in range(20):  # multiple random starts
    x0 = np.random.rand(6)
    res = minimize(lambda x: expected_improvement(x, gp, y_best),
                   x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_ei:
        best_ei = res.fun
        best_x = res.x

x_next = best_x
print("Next hyperparameter set to try:", x_next)


Next hyperparameter set to try: [0.88499754 0.00537971 0.51814368 0.5438813  0.21752085 0.68076504]


Week-02

In [39]:
X_init = data_in
y_init = data_out

x_new = np.array([[0.000000,0.210370,1.000000,0.000000,0.323749,1.000000]])
y_new = np.array([0.543430844426678])

X_init = np.vstack([X_init, x_new])
y_init = np.hstack([y_init, y_new])

y_best = y_init.max()  # best performance so far

#print(X_init)
#print(y_init)

In [40]:
# --- Fit Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_init)
y_best = y_init.max()  # still the best observed performance

In [41]:
def expected_improvement(x, gp, y_best, xi=0.05):
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    mu, sigma = mu[0], sigma[0]
    if sigma == 0:
        return 0
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # negative for minimize


In [42]:
bounds = [(0, 1)] * 6
best_x = None
best_ei = float('inf')

for _ in range(20):
    x0 = np.random.rand(6)
    res = minimize(lambda x: expected_improvement(x, gp, y_best),
                   x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_ei:
        best_ei = res.fun
        best_x = res.x

x_next = best_x
print("Next hyperparameter set to try:", x_next)


Next hyperparameter set to try: [0.05022557 0.60288852 0.27723461 0.32645993 0.1148645  0.62978107]


Week-03

In [43]:
X_init = np.vstack([
    data_in, 
    [0.000000,0.210370,1.000000,0.000000,0.323749,1.000000],
    [0.403152,0.365261,0.889930,0.311536,0.081832,0.228459]
])

y_init = np.hstack([
    data_out, 
    0.543430844426678,
    0.2524752216448303
])

# --- Update best observed performance ---
y_best = y_init.max()

# --- Fit Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_init)

# --- Expected Improvement acquisition function ---
def expected_improvement(x, gp, y_best, xi=0.05):  # increase xi for exploration
    x = np.array(x).reshape(1, -1)
    mu, sigma = gp.predict(x, return_std=True)
    mu, sigma = mu[0], sigma[0]
    if sigma == 0.0:
        return 0.0
    imp = mu - y_best - xi
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return -ei  # negative for minimization

# --- Optimize acquisition function ---
bounds = [(0, 1)] * 6
best_x = None
best_ei = float('inf')

# Use more random restarts for robust optimization
n_restarts = 50
for _ in range(n_restarts):
    x0 = np.random.rand(6)
    res = minimize(lambda x: expected_improvement(x, gp, y_best, xi=0.05),
                   x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_ei:
        best_ei = res.fun
        best_x = res.x

x_next = best_x
print("Next hyperparameter set to try:", x_next)

res_formatted = [f"{r:.6f}" for r in x_next]
result = "-".join(res_formatted)
print(result)

Next hyperparameter set to try: [0.35897668 0.92727929 0.08181819 0.16037732 0.96161762 0.86432363]
0.358977-0.927279-0.081818-0.160377-0.961618-0.864324


Week-04

In [45]:
X_init = np.array([
 [0.27262382, 0.32449536, 0.89710881, 0.83295115, 0.15406269, 0.79586362],
 [0.54300258, 0.9246939 , 0.34156746, 0.64648585, 0.71844033, 0.34313266],
 [0.09083225, 0.66152938, 0.06593091, 0.25857701, 0.96345285, 0.6402654 ],
 [0.11886697, 0.61505494, 0.90581639, 0.8553003 , 0.41363143, 0.58523563],
 [0.63021764, 0.8380969 , 0.68001305, 0.73189509, 0.52673671, 0.34842921],
 [0.76491917, 0.25588292, 0.60908422, 0.21807904, 0.32294277, 0.09579366],
 [0.05789554, 0.49167222, 0.24742222, 0.21811844, 0.42042833, 0.73096984],
 [0.19525188, 0.07922665, 0.55458046, 0.17056682, 0.01494418, 0.10703171],
 [0.64230298, 0.83687455, 0.02179269, 0.10148801, 0.68307083, 0.6924164 ],
 [0.78994255, 0.19554501, 0.57562333, 0.07365919, 0.25904917, 0.05109986],
 [0.52849733, 0.45742436, 0.36009569, 0.36204551, 0.81689098, 0.63747637],
 [0.72261522, 0.01181284, 0.06364591, 0.16517311, 0.07924415, 0.35995166],
 [0.07566492, 0.33450212, 0.13273274, 0.60831236, 0.91838592, 0.82233079],
 [0.94245084, 0.37743962, 0.48612233, 0.22879108, 0.08263175, 0.71195755],
 [0.14864702, 0.03394336, 0.72880565, 0.31606646, 0.02176938, 0.51691776],
 [0.81711239, 0.54816823, 0.10334758, 0.12436955, 0.72823482, 0.44967361],
 [0.41762629, 0.06409998, 0.24566877, 0.5590408 , 0.19153138, 0.25464092],
 [0.72628566, 0.46489581, 0.92457051, 0.8072454 , 0.6354384 , 0.14341787],
 [0.31981043, 0.52009759, 0.29067775, 0.87670668, 0.49503469, 0.6190825 ],
 [0.87987128, 0.39796199, 0.00363456, 0.95699064, 0.26451373, 0.11486924],
 [0.54124078, 0.63140314, 0.03190205, 0.44998156, 0.79865282, 0.63370429],
 [0.22634792, 0.11502581, 0.82474966, 0.94538372, 0.90531153, 0.95101392],
 [0.68685257, 0.04101721, 0.00757301, 0.285009  , 0.69156848, 0.6555429 ],
 [0.17597754, 0.6244165 , 0.29554198, 0.46955276, 0.09776977, 0.72814108],
 [0.88164674, 0.20445019, 0.41447436, 0.42038468, 0.26491501, 0.73066019],
 [0.06661051, 0.52804507, 0.8160952 , 0.96101714, 0.08650933, 0.77778822],
 [0.93246638, 0.48881189, 0.25860774, 0.95624344, 0.19042781, 0.51985176],
 [0.84686697, 0.14242917, 0.06066859, 0.75629213, 0.5523983 , 0.08130609],
 [0.80628208, 0.32412237, 0.72607601, 0.14871213, 0.7193764 , 0.36288398],
 [0.47682313, 0.34094195, 0.01433523, 0.88013956, 0.9986547 , 0.07966402]
])

y_init = np.array([
 0.6044327 , 0.56275307, 0.00750324, 0.0614243 , 0.2730468 , 0.08374657,
 1.3649683 , 0.09264495, 0.0178696 , 0.03356494, 0.0735163 , 0.2063097 ,
 0.00882563, 0.26840032, 0.61152553, 0.01479818, 0.27489251, 0.06676325,
 0.04211835, 0.00270147, 0.01820907, 0.00701603, 0.10050661, 0.47539552,
 0.67514163, 0.51645722, 0.00377748, 0.00313433, 0.02134252, 0.09541116
])

X_all = np.vstack([
X_init, 
[0.000000,0.210370,1.000000,0.000000,0.323749,1.000000],
[0.403152,0.365261,0.889930,0.311536,0.081832,0.228459],
[0.677250,0.098533,0.699430,0.859383,0.461203,0.197559]
])

y_all = np.hstack([
y_init, 
0.543430844426678,
0.2524752216448303,
0.00864398552164319
])


eps= 1e-20
signs = np.sign(y_all)
signs[signs == 0] = 1.0
y_trans = signs * np.log10(np.abs(y_all) + eps)


kernel = Matern(length_scale = 0.1, nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_all, y_trans)

def acquisition_ei(X, gp, y_best, xi=0.05):
    mu, sigma = gp.predict(X, return_std=True)
    sigma = sigma.reshape(-1, 1)
    mu = mu.reshape(-1, 1)
    imp = mu - y_best - xi
    Z = imp / (sigma + 1e-9)
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei.ravel()

grid_size = 100
margin = 0.03
n = 6

# Create 1D ranges
x1 = np.linspace(margin, 1-margin, n)
x2 = np.linspace(margin, 1-margin, n)
x3 = np.linspace(margin, 1-margin, n)
x4 = np.linspace(margin, 1-margin, n)
x5 = np.linspace(margin, 1-margin, n)
x6 = np.linspace(margin, 1-margin, n)

# Create 6D meshgrid
X1, X2, X3, X4, X5, X6 = np.meshgrid(
    x1, x2, x3, x4, x5, x6, 
    indexing='ij'
)

# Convert into candidate points
X_candidates = np.vstack([
    X1.ravel(),
    X2.ravel(),
    X3.ravel(),
    X4.ravel(),
    X5.ravel(),
    X6.ravel()
]).T

# --- 6. Compute EI across the grid ---
y_best = np.max(y_trans)
acq_values = acquisition_ei(X_candidates, gp, y_best, xi=1.0)

# --- 7. Select the next point ---
next = X_candidates[np.argmax(acq_values)]
best_ei = np.max(acq_values)

# --- 8. Display results with precision ---
print(f"[{next[0]:.6f}, {next[1]:.6f}, {next[2]:.6f}, {next[3]:.6f}, {next[4]:.6f}, {next[5]:.6f}]")

print(cf.format_inputdata(next))

[0.218000, 0.218000, 0.406000, 0.218000, 0.218000, 0.970000]
0.218000-0.218000-0.406000-0.218000-0.218000-0.970000
